In [ ]:
"""[Kaggle L4] PRECISION del 4B: la casilla vacia del argumento del umbral.

El log del campeon valida el RECALL del 4B (36 aciertos, todos con beam_score <= 0,486) pero
solo imprime lineas ALL_CORRECT: no dice nada de sus candidatos EQUIVOCADOS. Si el 4B produce
errores confiados bajo el umbral, la cascada y el cambio de umbral del DFS se caen.

Este kernel mide, sin TTT y en bf16 NATIVO de L4 (el intento en T4/fp16 del 21-ago fue NULO:
0/16 con gt_nll 5-28 donde el 2B saca 0,01-0,7 -- checkpoint bf16 + T4 sin soporte nativo),
las MISMAS 40 tareas de eval del banco 2B (ids embebidos: kernel offline, y comparabilidad
exacta). GATE al frente: si la tarea que el 2B clava con NLL 0,0 no baja de 1,0 en el 4B,
el modelo cargo mal y se aborta sin quemar la sesion.
"""
import glob
import json
import os
import time

import numpy as np
import torch
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM

T00 = time.time()
_SDPA = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
IM_START, USER, ASSIST, NL, EOS = 14, 11, 12, 10, 15
ARC_TOKENS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]
_TOKID = torch.tensor(ARC_TOKENS, dtype=torch.long, device="cuda:0")
THR = -float(np.log(0.2))

# Resolucion por BUSQUEDA (el layout del mount cambio dos veces; no fiarse de rutas fijas)
_COMP = None
for c in glob.glob("/kaggle/input/**/arc-agi_evaluation_challenges.json", recursive=True):
    _COMP = os.path.dirname(c)
    break
assert _COMP, "no se encontro la data de la competencia"
MDIR = None
for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
    d = os.path.dirname(c)
    if glob.glob(os.path.join(d, "*.safetensors")):
        MDIR = d
        break
assert MDIR, "no se encontro el modelo adjunto"
print(f"[setup] comp={_COMP} modelo={MDIR}", flush=True)

CHAL = json.load(open(f"{_COMP}/arc-agi_evaluation_challenges.json"))
SOLS = json.load(open(f"{_COMP}/arc-agi_evaluation_solutions.json"))

# Las MISMAS 40 del banco 2B (eval AGI-2, no-AGI1, orden por coste)
KEYS = ["e8686506", "20270e3b", "7b5033c1", "b0039139", "78332cb0", "28a6681f", "d35bdbdc",
        "dbff022c", "4c3d4a41", "446ef5d2", "dd6b8c4b", "3dc255db", "16de56c4", "97d7923e",
        "2b83f449", "bf45cf4b", "faa9f03d", "eee78d87", "31f7f899", "5dbc8537", "291dc1e1",
        "f931b4a8", "f560132c", "271d71e2", "247ef758", "7666fa5d", "21897d95", "1818057f",
        "5545f144", "135a2760", "1ae2feb7", "38007db0", "3a25b0d8", "581f7754", "6e453dd6",
        "65b59efc", "b9e38dc0", "53fb4810", "7ed72f31", "c7f57c3e"]
GATE = "38007db0"          # el 2B la clava con NLL 0,0; si el 4B no baja de 1,0, carga rota


def grid_tokens(g):
    out = []
    for i, row in enumerate(g):
        out.extend(int(c) for c in row)
        if i < len(g) - 1:
            out.append(NL)
    return out


def pair_tokens(inp, out):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + \
           [IM_START, ASSIST, NL] + grid_tokens(out) + [EOS]


def query_tokens(inp):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]


MODEL = AutoModelForCausalLM.from_pretrained(MDIR, torch_dtype=torch.bfloat16,
                                             device_map="cuda:0", attn_implementation="sdpa")
MODEL.eval()
MODEL.config.use_cache = True
print(f"[setup] 4B bf16 | mem={torch.cuda.memory_allocated()/1e9:.1f}GB "
      f"({time.time()-T00:.0f}s)", flush=True)


@torch.no_grad()
def prefill(toks, chunk=512):
    cache, out = None, None
    for i in range(0, len(toks), chunk):
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=torch.tensor([toks[i:i + chunk]], device="cuda:0",
                                               dtype=torch.long),
                        past_key_values=cache, use_cache=True, return_dict=True)
        cache = out.past_key_values
    return out.logits[:, -1], cache


@torch.no_grad()
def nll_gt(prompt, ans):
    last, cache = prefill(prompt)
    parts = [last.float()]
    if len(ans) > 1:
        with sdpa_kernel(_SDPA):
            o = MODEL(input_ids=torch.tensor([ans[:-1]], device="cuda:0", dtype=torch.long),
                      past_key_values=cache, use_cache=True, return_dict=True)
        parts.append(o.logits[0].float())
    lg = torch.cat([x.view(-1, x.shape[-1]) for x in parts], 0)
    v = -torch.log_softmax(lg, -1).gather(
        1, torch.tensor(ans, device="cuda:0").unsqueeze(1)).sum().item()
    del parts, lg, cache
    torch.cuda.empty_cache()
    return v


def toks_to_grid(toks):
    rows, row = [], []
    for t in toks:
        if t == NL:
            rows.append(row)
            row = []
        elif 0 <= t <= 9:
            row.append(t)
        else:
            return None
    if row:
        rows.append(row)
    if not rows or not rows[0] or any(len(r) != len(rows[0]) for r in rows):
        return None
    return rows


@torch.no_grad()
def dfs_scored(prompt, depth, budget=45):
    t0 = time.time()
    last, cache0 = prefill(prompt)
    leaves, path = [], []

    def rec(logits, score, pos, cache, left):
        if time.time() - t0 > budget or left <= 1:
            return
        lf = logits.float()
        nll = (score + torch.logsumexp(lf, dim=-1, keepdim=True)
               - lf.index_select(-1, _TOKID)).cpu()
        cands = []
        for j, t in enumerate(ARC_TOKENS):
            sc = nll[0, j].item()
            if sc < THR:
                if t == EOS:
                    leaves.append((sc, list(path)))
                else:
                    cands.append((sc, t))
        for sc, t in sorted(cands):
            if time.time() - t0 > budget:
                return
            with sdpa_kernel(_SDPA):
                o = MODEL(input_ids=torch.tensor([[t]], device="cuda:0", dtype=torch.long),
                          past_key_values=cache, use_cache=True, return_dict=True)
            path.append(t)
            rec(o.logits[:, -1], sc, pos + 1, o.past_key_values, left - 1)
            path.pop()

    rec(last, 0.0, len(prompt), cache0, depth)
    return leaves, time.time() - t0


def make_prompt(k):
    t = CHAL[k]
    p = []
    for x in t["train"]:
        p += pair_tokens(x["input"], x["output"])
    p += query_tokens(t["test"][0]["input"])
    return p


# GATE: aborta si el modelo cargo mal, sin quemar la sesion
_n = nll_gt(make_prompt(GATE), grid_tokens(SOLS[GATE][0]) + [EOS])
print(f"[gate] {GATE}: NLL de la verdad {_n:.4f}", flush=True)
assert _n < 1.0, f"gate FALLA ({_n:.3f}): modelo mal cargado, se aborta"

RES = {"tareas": [], "umbral": THR, "modelo": "4b-bf16-L4"}
for i, k in enumerate(KEYS):
    prompt = make_prompt(k)
    gt = SOLS[k][0]
    gt_toks = grid_tokens(gt) + [EOS]
    try:
        hojas, secs = dfs_scored(prompt, min(931, len(gt_toks) + 40))
    except torch.cuda.OutOfMemoryError:
        print(f"[{i+1}/{len(KEYS)}] {k}: OOM, se salta", flush=True)
        torch.cuda.empty_cache()
        continue
    hojas.sort(key=lambda x: x[0])
    grids = [(sc, toks_to_grid(p)) for sc, p in hojas]
    best_sc = grids[0][0] if grids else None
    best_ok = bool(grids and grids[0][1] == gt)
    dos_ok = bool(any(g == gt for _, g in grids[:2]))
    n = nll_gt(prompt, gt_toks)
    RES["tareas"].append({
        "task": k, "L": len(gt_toks), "pool": len(grids),
        "best_score": None if best_sc is None else round(best_sc, 4),
        "best_ok": best_ok, "pass2_ok": dos_ok,
        "gt_nll": round(n, 4), "decode_s": round(secs, 1),
        "scores": [round(s, 4) for s, _ in grids[:5]]})
    print(f"[{i+1}/{len(KEYS)}] {k}: pool {len(grids)} | best "
          f"{'-' if best_sc is None else f'{best_sc:.4f}'} | ok {best_ok} | "
          f"gt_nll {n:.3f} | {secs:.0f}s", flush=True)
    json.dump(RES, open("/kaggle/working/tier1_4b.json", "w"), indent=1)
    torch.cuda.empty_cache()

con = [r for r in RES["tareas"] if r["best_score"] is not None]
barrido = []
for tau in (0.05, 0.1, 0.2, 0.5, 0.75, 1.0, THR):
    ret = [r for r in con if r["best_score"] < tau]
    ok = sum(r["best_ok"] for r in ret)
    barrido.append({"tau": round(tau, 3), "retiradas": len(ret), "correctas": ok,
                    "precision": round(ok / len(ret), 3) if ret else None})
RES["barrido_tau"] = barrido
json.dump(RES, open("/kaggle/working/tier1_4b.json", "w"), indent=1)
print("RESULT_JSON:" + json.dumps({"barrido_tau": barrido, "n": len(RES["tareas"]),
                                   "resueltas": sum(r["best_ok"] for r in RES["tareas"]),
                                   "pass2": sum(r["pass2_ok"] for r in RES["tareas"])}),
      flush=True)
print(f"[fin] {time.time()-T00:.0f}s", flush=True)
